# Phase 03 — Psychological Capital → Employability

## Step 10 — Evidence-Grounded Recommendation System

මෙම notebook එකේ අරමුණ final Psychological Capital model එකෙන් ලබාගන්නා prediction සහ explainability evidence භාවිත කර practical, personalized සහ user-friendly development recommendations generate කිරීමයි.

Recommendation system එක final prediction model එක වෙනස් නොකරයි සහ model එක නැවත train නොකරයි.

Recommendations සකස් කිරීම සඳහා පහත evidence layers භාවිත කරයි:

1. Userගේ Hope, Optimism, Resilience සහ Self-Efficacy levels
2. Research dataset එකෙන් ලැබෙන benchmark values
3. Final model explainability evidence
4. Individual questionnaire-item responses

Technical model evidence backend decision-making සඳහා භාවිත කරන නමුත් user-facing recommendation language simple සහ professional ලෙස ඉදිරිපත් කරයි.

මෙම system එක automated hiring decision system එකක් නොව professional development guidance system එකක් ලෙස design කරයි.

In [1]:
from pathlib import Path

import json
import hashlib

import numpy as np
import pandas as pd
import joblib
import shap

d:\internship-employability-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "phase3_psycap":
    PROJECT_DIR = PROJECT_DIR.parent.parent

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "phase3_psycap_prepared.csv"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "phase3"
)

MODEL_PATH = (
    MODEL_DIR
    / "phase3_psycap_selected_model.pkl"
)

METADATA_PATH = (
    MODEL_DIR
    / "phase3_psycap_model_metadata.json"
)

FEATURE_SCHEMA_PATH = (
    MODEL_DIR
    / "phase3_psycap_feature_schema.json"
)

REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "recommendations"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("MODEL_PATH:", MODEL_PATH)
print("REPORT_DIR:", REPORT_DIR)

MODEL_PATH: d:\internship-employability-project\models\phase3\phase3_psycap_selected_model.pkl
REPORT_DIR: d:\internship-employability-project\reports\phase3\recommendations


In [3]:
final_model = joblib.load(
    MODEL_PATH
)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    metadata = json.load(file)

with open(
    FEATURE_SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as file:

    feature_schema = json.load(file)

print(
    "Loaded final model:",
    metadata["model_name"]
)

print(
    "Features:",
    feature_schema["features"]
)

Loaded final model: ElasticNet
Features: ['Hope_avg', 'Optimism_avg', 'Resilience_avg', 'SelfEfficacy_avg']


Verify model integrity

In [4]:
def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        for block in iter(
            lambda: file.read(4096),
            b""
        ):
            sha256.update(block)

    return sha256.hexdigest()


current_hash = calculate_sha256(
    MODEL_PATH
)

saved_hash = metadata[
    "model_sha256"
]

print(
    "Model hash valid:",
    current_hash == saved_hash
)

Model hash valid: True


In [5]:
df = pd.read_csv(
    DATA_PATH
)

FEATURES = [
    "Hope_avg",
    "Optimism_avg",
    "Resilience_avg",
    "SelfEfficacy_avg"
]

TARGET = "EVS_avg"

ITEM_FEATURES = [
    f"PsyCap{i}"
    for i in range(1, 13)
]

print(
    "Dataset shape:",
    df.shape
)



Dataset shape: (1000, 41)


## Psychological Capital Construct Mapping

Recommendation engine එකේ model-level construct evidence සහ item-level questionnaire evidence connect කිරීම සඳහා each Psychological Capital construct එකට අදාල items map කරයි.

Item-level evidence භාවිත කිරීමෙන් generic advice වෙනුවට user's actual response pattern එකට සම්බන්ධ specific development guidance ලබාදිය හැක.

In [6]:
CONSTRUCT_ITEMS = {

    "Hope_avg": [
        "PsyCap1",
        "PsyCap2",
        "PsyCap3"
    ],

    "Optimism_avg": [
        "PsyCap4",
        "PsyCap5",
        "PsyCap6"
    ],

    "Resilience_avg": [
        "PsyCap7",
        "PsyCap8",
        "PsyCap9"
    ],

    "SelfEfficacy_avg": [
        "PsyCap10",
        "PsyCap11",
        "PsyCap12"
    ]
}

Friendly construct labels

In [7]:
CONSTRUCT_LABELS = {

    "Hope_avg":
        "Hope",

    "Optimism_avg":
        "Optimism",

    "Resilience_avg":
        "Resilience",

    "SelfEfficacy_avg":
        "Self-Efficacy"
}

Item labels

In [8]:
ITEM_LABELS = {

    "PsyCap1":
        "finding ways to overcome obstacles",

    "PsyCap2":
        "trying different approaches to solve problems",

    "PsyCap3":
        "staying committed to personal goals",

    "PsyCap4":
        "expecting positive results from learning and development",

    "PsyCap5":
        "believing current effort can create future benefits",

    "PsyCap6":
        "learning something useful from success and failure",

    "PsyCap7":
        "analysing problems before responding",

    "PsyCap8":
        "identifying reasons for poor results and improving",

    "PsyCap9":
        "recovering and continuing after setbacks",

    "PsyCap10":
        "handling difficult tasks confidently",

    "PsyCap11":
        "asking for support when needed",

    "PsyCap12":
        "using personal abilities to make a useful contribution"
}

## Practical Recommendation Library

Each questionnaire item එකට practical action guidance එකක් map කරයි.

Recommendations:

- simple language භාවිත කරයි
- professional context එකට suitable වේ
- negative labels භාවිත නොකරයි
- immediately actionable steps ලබාදෙයි

Recommendation messages model evidence වෙනස් නොකරන අතර validated evidence user-friendly wording එකකට translate කරයි.

Recommendation library

In [9]:
ACTION_LIBRARY = {

    "PsyCap1": {
        "title":
            "Plan more than one way forward",

        "action":
            "When you face a difficult task, write down two or three possible ways to handle it before choosing the best option."
    },

    "PsyCap2": {
        "title":
            "Try a different approach",

        "action":
            "If your first solution does not work, try another method or ask someone experienced for a different perspective."
    },

    "PsyCap3": {
        "title":
            "Turn goals into small actions",

        "action":
            "Break an important career or work goal into small weekly tasks and check your progress regularly."
    },

    "PsyCap4": {
        "title":
            "Notice your progress",

        "action":
            "Keep a short record of skills, tasks or responsibilities you are improving. This can help you see progress more clearly."
    },

    "PsyCap5": {
        "title":
            "Connect today's effort with your future",

        "action":
            "Choose one professional-development activity and write down how it could support your future career or work goals."
    },

    "PsyCap6": {
        "title":
            "Learn from every result",

        "action":
            "After an important task, note one thing that worked well and one thing you would improve next time."
    },

    "PsyCap7": {
        "title":
            "Understand the problem first",

        "action":
            "Before reacting to a difficult situation, identify the main problem, possible causes and the information you need."
    },

    "PsyCap8": {
        "title":
            "Use feedback to improve",

        "action":
            "When a task does not go as planned, identify the main reason and choose one clear improvement for your next attempt."
    },

    "PsyCap9": {
        "title":
            "Build a recovery routine",

        "action":
            "After a setback, decide on one realistic next step and restart quickly instead of leaving the task unfinished."
    },

    "PsyCap10": {
        "title":
            "Build confidence through practice",

        "action":
            "Choose a slightly challenging task, complete it step by step and record what you managed successfully."
    },

    "PsyCap11": {
        "title":
            "Ask for support early",

        "action":
            "When you are stuck, ask a manager, mentor, colleague, lecturer or trainer for specific guidance instead of waiting too long."
    },

    "PsyCap12": {
        "title":
            "Use your strengths to contribute",

        "action":
            "Look for a small task where you can use one of your strongest skills to help your team, project or workplace."
    }
}

Data-Driven Benchmarks

## Data-Driven Development Benchmarks

Userගේ construct level එක arbitrary cut-off එකකට compare කිරීම වෙනුවට research dataset එකේ empirical distribution භාවිත කර benchmark values සකස් කරයි.

Each construct සඳහා:

- 25th percentile
- median
- 75th percentile

calculate කරයි.

මෙම benchmarks userගේ score research sample එකට සාපේක්ෂව interpret කිරීමට උපකාරී වන අතර hiring threshold එකක් ලෙස භාවිත නොකරයි.

Construct benchmarks

In [10]:
benchmark_rows = []

for feature in FEATURES:

    benchmark_rows.append({

        "Feature":
            feature,

        "Q25":
            df[feature].quantile(
                0.25
            ),

        "Median":
            df[feature].median(),

        "Q75":
            df[feature].quantile(
                0.75
            ),

        "Mean":
            df[feature].mean()
    })


construct_benchmark_df = pd.DataFrame(
    benchmark_rows
)

construct_benchmark_df

,Feature,Q25,Median,Q75,Mean
0,Hope_avg,4.0,4.333333,4.333333,4.236000
1,Optimism_avg,4.0,4.333333,4.333333,4.271667
2,Resilience_avg,4.0,4.333333,4.333333,4.141000
3,SelfEfficacy_avg,4.0,4.000000,4.333333,4.139667


Load Saved Model Components for Local SHAP

In [11]:
scaler = final_model.named_steps[
    "scaler"
]

elasticnet_model = final_model.named_steps[
    "model"
]

X_reference = df[
    FEATURES
]

X_reference_scaled = pd.DataFrame(
    scaler.transform(
        X_reference
    ),
    columns=FEATURES,
    index=X_reference.index
)

shap_explainer = shap.LinearExplainer(
    elasticnet_model,
    X_reference_scaled
)

Background dataset has 1000 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1000 when initializing the masker.


Global SHAP Weights

In [12]:
GLOBAL_SHAP_IMPORTANCE = {

    "SelfEfficacy_avg":
        41.966880,

    "Resilience_avg":
        34.547883,

    "Optimism_avg":
        12.630413,

    "Hope_avg":
        10.854824
}

Preferred automatic loading

In [13]:
SHAP_REPORT_PATH = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "shap_explainability"
    / "SHAP_Normalized_Importance.csv"
)

if SHAP_REPORT_PATH.exists():

    global_shap_df = pd.read_csv(
        SHAP_REPORT_PATH
    )

    GLOBAL_SHAP_IMPORTANCE = dict(
        zip(
            global_shap_df["Feature"],
            global_shap_df[
                "Relative_SHAP_Percent"
            ]
        )
    )

print(
    GLOBAL_SHAP_IMPORTANCE
)

{'SelfEfficacy_avg': 41.96687997271973, 'Resilience_avg': 34.547882724046666, 'Optimism_avg': 12.630413435296738, 'Hope_avg': 10.854823867936888}


Input Validation

In [14]:
def validate_psycap_input(
    responses
):

    required_items = [
        f"PsyCap{i}"
        for i in range(
            1,
            13
        )
    ]

    missing_items = [
        item
        for item in required_items
        if item not in responses
    ]

    if missing_items:
        raise ValueError(
            f"Missing items: {missing_items}"
        )

    invalid_items = {}

    for item in required_items:

        value = responses[item]

        if not isinstance(
            value,
            (int, float, np.integer, np.floating)
        ):
            invalid_items[item] = value

        elif value < 1 or value > 5:
            invalid_items[item] = value

    if invalid_items:

        raise ValueError(
            f"Values must be between 1 and 5: {invalid_items}"
        )

    return True

Calculate constructs

In [15]:
def calculate_constructs(
    responses
):

    constructs = {}

    for construct, items in (
        CONSTRUCT_ITEMS.items()
    ):

        constructs[
            construct
        ] = float(
            np.mean(
                [
                    responses[item]
                    for item in items
                ]
            )
        )

    return constructs

Local SHAP explanation

In [16]:
def calculate_local_shap(
    construct_values
):

    user_X = pd.DataFrame(
        [
            construct_values
        ],
        columns=FEATURES
    )

    user_scaled = pd.DataFrame(
        scaler.transform(
            user_X
        ),
        columns=FEATURES
    )

    user_shap = shap_explainer(
        user_scaled
    )

    shap_map = {
        feature:
            float(
                user_shap.values[
                    0,
                    index
                ]
            )
        for index, feature
        in enumerate(FEATURES)
    }

    return shap_map

Determine user development status

In [17]:
def get_construct_status(
    feature,
    value
):

    benchmark = (
        construct_benchmark_df[
            construct_benchmark_df[
                "Feature"
            ]
            ==
            feature
        ]
        .iloc[0]
    )

    q25 = benchmark["Q25"]
    median = benchmark["Median"]
    q75 = benchmark["Q75"]

    if value < q25:

        return "Development Priority"

    elif value < median:

        return "Growth Opportunity"

    elif value >= q75:

        return "Strong Area"

    else:

        return "Developing Well"

## Internal Recommendation Prioritization

Recommendation priority එක single factor එකකින් determine නොකරයි.

System එක considers:

1. user's current construct position relative to sample benchmark
2. final model global importance
3. user's local SHAP contribution
4. lowest item-level response within the construct

Internal priority calculations user-facing score ලෙස display නොකරයි. ඒවා recommendation order determine කිරීම සඳහා පමණක් භාවිත කරයි.

Priority analysis

In [18]:
def build_priority_analysis(
    responses,
    constructs,
    local_shap
):

    rows = []

    for feature in FEATURES:

        value = constructs[
            feature
        ]

        benchmark = (
            construct_benchmark_df[
                construct_benchmark_df[
                    "Feature"
                ]
                ==
                feature
            ]
            .iloc[0]
        )

        median = float(
            benchmark["Median"]
        )

        gap_from_median = max(
            0.0,
            median - value
        )

        items = CONSTRUCT_ITEMS[
            feature
        ]

        item_values = {
            item:
                responses[item]
            for item in items
        }

        lowest_item = min(
            item_values,
            key=item_values.get
        )

        lowest_item_value = (
            item_values[
                lowest_item
            ]
        )

        global_weight = (
            GLOBAL_SHAP_IMPORTANCE[
                feature
            ]
            / 100
        )

        local_negative_contribution = max(
            0.0,
            -local_shap[
                feature
            ]
        )

        priority_score = (
            gap_from_median
            * global_weight
        ) + (
            0.25
            * local_negative_contribution
        )

        rows.append({

            "Feature":
                feature,

            "Construct":
                CONSTRUCT_LABELS[
                    feature
                ],

            "Value":
                value,

            "Median_Benchmark":
                median,

            "Status":
                get_construct_status(
                    feature,
                    value
                ),

            "Global_Importance_Percent":
                GLOBAL_SHAP_IMPORTANCE[
                    feature
                ],

            "Local_SHAP":
                local_shap[
                    feature
                ],

            "Lowest_Item":
                lowest_item,

            "Lowest_Item_Value":
                lowest_item_value,

            "Priority_Score_Internal":
                priority_score
        })


    priority_df = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "Priority_Score_Internal",
                "Global_Importance_Percent"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return priority_df

Select strengths

In [19]:
def select_strengths(
    priority_df,
    max_strengths=2
):

    strength_candidates = (
        priority_df[
            priority_df[
                "Status"
            ].isin(
                [
                    "Strong Area",
                    "Developing Well"
                ]
            )
        ]
        .sort_values(
            [
                "Value",
                "Global_Importance_Percent"
            ],
            ascending=[
                False,
                False
            ]
        )
    )

    if len(
        strength_candidates
    ) == 0:

        strength_candidates = (
            priority_df
            .sort_values(
                "Value",
                ascending=False
            )
        )

    return (
        strength_candidates
        .head(
            max_strengths
        )
        .copy()
    )

Select development priorities

In [20]:
def select_development_priorities(
    priority_df,
    max_priorities=2
):

    priority_candidates = (
        priority_df[
            priority_df[
                "Status"
            ].isin(
                [
                    "Development Priority",
                    "Growth Opportunity"
                ]
            )
        ]
        .sort_values(
            [
                "Priority_Score_Internal",
                "Global_Importance_Percent"
            ],
            ascending=[
                False,
                False
            ]
        )
    )

    if len(
        priority_candidates
    ) == 0:

        priority_candidates = (
            priority_df
            .sort_values(
                [
                    "Value",
                    "Global_Importance_Percent"
                ],
                ascending=[
                    True,
                    False
                ]
            )
        )

    return (
        priority_candidates
        .head(
            max_priorities
        )
        .copy()
    )

User-Friendly Strength Messages

In [21]:
def create_strength_messages(
    strengths_df
):

    messages = []

    for _, row in (
        strengths_df.iterrows()
    ):

        construct = row[
            "Construct"
        ]

        if construct == "Self-Efficacy":

            message = (
                "You show good confidence in using your abilities and handling responsibilities. "
                "Keep building this by taking on meaningful challenges and recognising what you complete successfully."
            )

        elif construct == "Resilience":

            message = (
                "You show a good ability to recover and continue when things do not go as planned. "
                "Keep using feedback and experience to move forward."
            )

        elif construct == "Optimism":

            message = (
                "You generally maintain a positive view of your progress and future development. "
                "Keep connecting your current effort with clear career goals."
            )

        else:

            message = (
                "You show a good ability to stay focused on goals and find ways forward. "
                "Keep turning important goals into practical steps."
            )

        messages.append({
            "Construct":
                construct,

            "Message":
                message
        })

    return messages

User-Friendly Development Recommendation

In [22]:
def create_development_recommendations(
    priorities_df
):

    recommendations = []

    for _, row in (
        priorities_df.iterrows()
    ):

        construct = row[
            "Construct"
        ]

        item = row[
            "Lowest_Item"
        ]

        action_info = (
            ACTION_LIBRARY[
                item
            ]
        )

        item_focus = (
            ITEM_LABELS[
                item
            ]
        )

        reason = (
            f"{construct} is a useful area to focus on next. "
            f"Your responses suggest that strengthening {item_focus} may support your professional development."
        )

        recommendations.append({

            "Priority":
                construct,

            "Why_this_was_selected":
                reason,

            "Action_Title":
                action_info[
                    "title"
                ],

            "Action":
                action_info[
                    "action"
                ]
        })

    return recommendations

Employability prediction

In [23]:
def predict_employability(
    constructs
):

    user_X = pd.DataFrame(
        [
            constructs
        ],
        columns=FEATURES
    )

    prediction = float(
        final_model.predict(
            user_X
        )[0]
    )

    return prediction

Complete Recommendation Engine

In [24]:
def generate_recommendation_report(
    responses
):

    validate_psycap_input(
        responses
    )

    constructs = calculate_constructs(
        responses
    )

    prediction = predict_employability(
        constructs
    )

    local_shap = calculate_local_shap(
        constructs
    )

    priority_analysis = (
        build_priority_analysis(
            responses,
            constructs,
            local_shap
        )
    )

    strengths_df = (
        select_strengths(
            priority_analysis
        )
    )

    priorities_df = (
        select_development_priorities(
            priority_analysis
        )
    )

    strength_messages = (
        create_strength_messages(
            strengths_df
        )
    )

    development_recommendations = (
        create_development_recommendations(
            priorities_df
        )
    )

    report = {

        "prediction":
            prediction,

        "constructs":
            constructs,

        "strengths":
            strength_messages,

        "development_priorities":
            development_recommendations,

        "internal_evidence":
            priority_analysis
            .to_dict(
                orient="records"
            )
    }

    return report

Test with a realistic user

In [25]:
sample_user = {

    "PsyCap1": 4,
    "PsyCap2": 4,
    "PsyCap3": 4,

    "PsyCap4": 4,
    "PsyCap5": 4,
    "PsyCap6": 4,

    "PsyCap7": 3,
    "PsyCap8": 3,
    "PsyCap9": 4,

    "PsyCap10": 3,
    "PsyCap11": 2,
    "PsyCap12": 4
}


sample_report = (
    generate_recommendation_report(
        sample_user
    )
)

sample_report

{'prediction': 3.5247280270058385,
 'constructs': {'Hope_avg': 4.0,
  'Optimism_avg': 4.0,
  'Resilience_avg': 3.3333333333333335,
  'SelfEfficacy_avg': 3.0},
 'strengths': [{'Construct': 'Hope',
   'Message': 'You show a good ability to stay focused on goals and find ways forward. Keep turning important goals into practical steps.'},
  {'Construct': 'Optimism',
   'Message': 'You generally maintain a positive view of your progress and future development. Keep connecting your current effort with clear career goals.'}],
 'development_priorities': [{'Priority': 'Self-Efficacy',
   'Why_this_was_selected': 'Self-Efficacy is a useful area to focus on next. Your responses suggest that strengthening asking for support when needed may support your professional development.',
   'Action_Title': 'Ask for support early',
   'Action': 'When you are stuck, ask a manager, mentor, colleague, lecturer or trainer for specific guidance instead of waiting too long.'},
  {'Priority': 'Resilience',
   'Wh

Create clean user-facing report

In [28]:
def format_user_friendly_report(report):

    output = []

    output.append(
        f"Predicted Employability: "
        f"{report['prediction']:.2f}"
    )

    output.append("")
    output.append("Strengths")

    for strength in report["strengths"]:

        output.append(
            f"- {strength['Construct']}: "
            f"{strength['Message']}"
        )

    output.append("")
    output.append("Development Areas")

    for recommendation in (
        report["development_priorities"]
    ):

        output.append(
            f"- {recommendation['Priority']}"
        )

    output.append("")
    output.append("Recommendations")

    for index, recommendation in enumerate(
        report["development_priorities"],
        start=1
    ):

        output.append(
            f"{index}. {recommendation['Priority']}"
        )

        output.append(
            f"   {recommendation['Action']}"
        )

    return "\n".join(output)

In [29]:
friendly_output = (
    format_user_friendly_report(
        sample_report
    )
)

print(
    friendly_output
)

Predicted Employability: 3.52

Strengths
- Hope: You show a good ability to stay focused on goals and find ways forward. Keep turning important goals into practical steps.
- Optimism: You generally maintain a positive view of your progress and future development. Keep connecting your current effort with clear career goals.

Development Areas
- Self-Efficacy
- Resilience

Recommendations
1. Self-Efficacy
   When you are stuck, ask a manager, mentor, colleague, lecturer or trainer for specific guidance instead of waiting too long.
2. Resilience
   Before reacting to a difficult situation, identify the main problem, possible causes and the information you need.


In [30]:
FORBIDDEN_USER_TERMS = [
    "SHAP",
    "ElasticNet",
    "coefficient",
    "hyperparameter",
    "standard deviation",
    "weakness",
    "low score",
    "negative contribution"
]


def validate_user_friendly_language(
    text
):

    found_terms = [
        term
        for term in FORBIDDEN_USER_TERMS
        if term.lower()
        in text.lower()
    ]

    return {
        "User_Friendly":
            len(found_terms) == 0,

        "Technical_or_Negative_Terms":
            found_terms
    }


user_language_validation = (
    validate_user_friendly_language(
        friendly_output
    )
)

user_language_validation

{'User_Friendly': True, 'Technical_or_Negative_Terms': []}

Recommendation Traceability

In [31]:
def create_recommendation_traceability(
    report
):

    evidence_df = pd.DataFrame(
        report[
            "internal_evidence"
        ]
    )

    trace_rows = []

    for recommendation in (
        report[
            "development_priorities"
        ]
    ):

        construct = recommendation[
            "Priority"
        ]

        row = evidence_df[
            evidence_df[
                "Construct"
            ]
            ==
            construct
        ].iloc[0]

        trace_rows.append({

            "Recommendation_Construct":
                construct,

            "User_Construct_Value":
                row[
                    "Value"
                ],

            "Benchmark_Median":
                row[
                    "Median_Benchmark"
                ],

            "Global_Model_Importance_Percent":
                row[
                    "Global_Importance_Percent"
                ],

            "Local_SHAP":
                row[
                    "Local_SHAP"
                ],

            "Supporting_Item":
                row[
                    "Lowest_Item"
                ],

            "Supporting_Item_Value":
                row[
                    "Lowest_Item_Value"
                ],

            "Recommendation_Action":
                recommendation[
                    "Action"
                ]
        })

    return pd.DataFrame(
        trace_rows
    )

In [32]:
traceability_df = (
    create_recommendation_traceability(
        sample_report
    )
)

traceability_df

,Recommendation_Construct,User_Construct_Value,Benchmark_Median,Global_Model_Importance_Percent,Local_SHAP,Supporting_Item,Supporting_Item_Value,Recommendation_Action
0,Self-Efficacy,3.000000,4.000000,41.966880,-0.310461,PsyCap11,2,"When you are stuck, ask a manager, mentor, col..."
1,Resilience,3.333333,4.333333,34.547883,-0.193598,PsyCap7,3,"Before reacting to a difficult situation, iden..."


Boundary Testing

In [33]:
low_user = {
    f"PsyCap{i}": 1
    for i in range(1, 13)
}

middle_user = {
    f"PsyCap{i}": 3
    for i in range(1, 13)
}

high_user = {
    f"PsyCap{i}": 5
    for i in range(1, 13)
}


boundary_reports = {

    "Low":
        generate_recommendation_report(
            low_user
        ),

    "Middle":
        generate_recommendation_report(
            middle_user
        ),

    "High":
        generate_recommendation_report(
            high_user
        )
}

Check logical prediction order

In [34]:
boundary_prediction_df = pd.DataFrame({

    "Case": [
        "Low",
        "Middle",
        "High"
    ],

    "Prediction": [
        boundary_reports[
            "Low"
        ][
            "prediction"
        ],

        boundary_reports[
            "Middle"
        ][
            "prediction"
        ],

        boundary_reports[
            "High"
        ][
            "prediction"
        ]
    ]
})

boundary_prediction_df

,Case,Prediction
0,Low,1.851498
1,Middle,3.259239
2,High,4.666980


In [35]:
logical_order = (
    boundary_prediction_df.loc[
        0,
        "Prediction"
    ]
    <
    boundary_prediction_df.loc[
        1,
        "Prediction"
    ]
    <
    boundary_prediction_df.loc[
        2,
        "Prediction"
    ]
)

print(
    "Logical prediction order:",
    logical_order
)

Logical prediction order: True


Invalid input tests

In [36]:
invalid_input_tests = []


try:

    invalid_case = sample_user.copy()

    invalid_case[
        "PsyCap1"
    ] = 6

    generate_recommendation_report(
        invalid_case
    )

    invalid_input_tests.append(
        {
            "Test":
                "Value above 5 rejected",

            "Status":
                "FAIL"
        }
    )

except ValueError:

    invalid_input_tests.append(
        {
            "Test":
                "Value above 5 rejected",

            "Status":
                "PASS"
        }
    )


try:

    missing_case = (
        sample_user.copy()
    )

    del missing_case[
        "PsyCap12"
    ]

    generate_recommendation_report(
        missing_case
    )

    invalid_input_tests.append(
        {
            "Test":
                "Missing item rejected",

            "Status":
                "FAIL"
        }
    )

except ValueError:

    invalid_input_tests.append(
        {
            "Test":
                "Missing item rejected",

            "Status":
                "PASS"
        }
    )


invalid_input_test_df = pd.DataFrame(
    invalid_input_tests
)

invalid_input_test_df

,Test,Status
0,Value above 5 rejected,PASS
1,Missing item rejected,PASS


Determinism Test

In [37]:
report_a = (
    generate_recommendation_report(
        sample_user
    )
)

report_b = (
    generate_recommendation_report(
        sample_user
    )
)


same_prediction = np.isclose(
    report_a[
        "prediction"
    ],
    report_b[
        "prediction"
    ]
)

same_priorities = (
    report_a[
        "development_priorities"
    ]
    ==
    report_b[
        "development_priorities"
    ]
)

print(
    "Same prediction:",
    same_prediction
)

print(
    "Same recommendations:",
    same_priorities
)

Same prediction: True
Same recommendations: True


Save Reports

In [38]:
construct_benchmark_df.to_csv(
    REPORT_DIR
    / "Construct_Benchmarks.csv",
    index=False
)

traceability_df.to_csv(
    REPORT_DIR
    / "Sample_Recommendation_Traceability.csv",
    index=False
)

boundary_prediction_df.to_csv(
    REPORT_DIR
    / "Recommendation_Boundary_Tests.csv",
    index=False
)

invalid_input_test_df.to_csv(
    REPORT_DIR
    / "Recommendation_Invalid_Input_Tests.csv",
    index=False
)

Save sample evidence

In [39]:
sample_internal_evidence_df = pd.DataFrame(
    sample_report[
        "internal_evidence"
    ]
)

sample_internal_evidence_df.to_csv(
    REPORT_DIR
    / "Sample_Recommendation_Internal_Evidence.csv",
    index=False
)

In [40]:
step10_validation = pd.DataFrame({

    "Check": [

        "Exact saved model loaded",

        "Model integrity verified",

        "No model retraining performed",

        "Construct calculation implemented",

        "Dataset-based benchmarks calculated",

        "Local SHAP evidence incorporated",

        "Global SHAP evidence incorporated",

        "Item-level evidence incorporated",

        "Strength identification implemented",

        "Development-priority identification implemented",

        "Specific actionable guidance generated",

        "Technical terminology removed from user output",

        "Recommendation traceability verified",

        "Low boundary case tested",

        "Middle boundary case tested",

        "High boundary case tested",

        "Logical prediction ordering verified",

        "Invalid values rejected",

        "Missing inputs rejected",

        "Deterministic recommendations verified"
    ],

    "Status": [

        "PASS",

        (
            "PASS"
            if current_hash
            ==
            saved_hash
            else "FAIL"
        ),

        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",

        (
            "PASS"
            if user_language_validation[
                "User_Friendly"
            ]
            else "REVIEW"
        ),

        "PASS",
        "PASS",
        "PASS",
        "PASS",

        (
            "PASS"
            if logical_order
            else "FAIL"
        ),

        (
            invalid_input_test_df.iloc[
                0
            ]["Status"]
        ),

        (
            invalid_input_test_df.iloc[
                1
            ]["Status"]
        ),

        (
            "PASS"
            if same_prediction
            and same_priorities
            else "FAIL"
        )
    ]
})

step10_validation

,Check,Status
0,Exact saved model loaded,PASS
1,Model integrity verified,PASS
2,No model retraining performed,PASS
3,Construct calculation implemented,PASS
4,Dataset-based benchmarks calculated,PASS
5,Local SHAP evidence incorporated,PASS
6,Global SHAP evidence incorporated,PASS
7,Item-level evidence incorporated,PASS
8,Strength identification implemented,PASS
9,Development-priority identification implemented,PASS


In [41]:
step10_validation.to_csv(
    REPORT_DIR
    / "Step10_Validation.csv",
    index=False
)

step10_validation

,Check,Status
0,Exact saved model loaded,PASS
1,Model integrity verified,PASS
2,No model retraining performed,PASS
3,Construct calculation implemented,PASS
4,Dataset-based benchmarks calculated,PASS
5,Local SHAP evidence incorporated,PASS
6,Global SHAP evidence incorporated,PASS
7,Item-level evidence incorporated,PASS
8,Strength identification implemented,PASS
9,Development-priority identification implemented,PASS


In [42]:
print(
    format_user_friendly_report(
        sample_report
    )
)

Predicted Employability: 3.52

Strengths
- Hope: You show a good ability to stay focused on goals and find ways forward. Keep turning important goals into practical steps.
- Optimism: You generally maintain a positive view of your progress and future development. Keep connecting your current effort with clear career goals.

Development Areas
- Self-Efficacy
- Resilience

Recommendations
1. Self-Efficacy
   When you are stuck, ask a manager, mentor, colleague, lecturer or trainer for specific guidance instead of waiting too long.
2. Resilience
   Before reacting to a difficult situation, identify the main problem, possible causes and the information you need.


## Step 10 අවසාන නිගමනය

Final Psychological Capital model සහ Step 09 explainability evidence භාවිත කර deterministic, evidence-grounded recommendation system එකක් සකස් කරන ලදී.

Recommendation engine එක final prediction model එක retrain නොකර exact saved artifact එක භාවිත කරයි.

User development priorities හඳුනා ගැනීම සඳහා construct-level performance, research-sample benchmarks, global model importance, local model explanation සහ item-level questionnaire responses combine කරන ලදී.

Generic recommendations වෙනුවට user's actual response pattern එකෙන් හඳුනාගත් relevant questionnaire item එක practical development action එකකට map කරන ලදී.

Supervisor feedback අනුව user-facing output එක technical terminology සහ negative labels වලින් තොරව simple professional language භාවිත කර නැවත සකස් කරන ලදී.

Recommendation system එක strengths, development priorities, reasons සහ short actionable next steps ලෙස output ලබාදෙන අතර every recommendation backend evidence එකට trace කළ හැක.

මෙම recommendations professional-development support සඳහා වන අතර automated hiring decisions සඳහා භාවිත කිරීමට අදහස් කර නැත.

In [43]:
expected_items = {
    f"PsyCap{i}"
    for i in range(1, 13)
}

mapped_items = set(
    ACTION_LIBRARY.keys()
)

coverage_complete = (
    expected_items
    ==
    mapped_items
)

print(
    "All PsyCap items have recommendation actions:",
    coverage_complete
)

All PsyCap items have recommendation actions: True


In [44]:
recommendation_coverage_df = pd.DataFrame({
    "Item": sorted(expected_items),
    "Has_Action": [
        item in ACTION_LIBRARY
        for item in sorted(expected_items)
    ],
    "Has_Label": [
        item in ITEM_LABELS
        for item in sorted(expected_items)
    ]
})

recommendation_coverage_df

,Item,Has_Action,Has_Label
0,PsyCap1,True,True
1,PsyCap10,True,True
2,PsyCap11,True,True
3,PsyCap12,True,True
4,PsyCap2,True,True
5,PsyCap3,True,True
6,PsyCap4,True,True
7,PsyCap5,True,True
8,PsyCap6,True,True
9,PsyCap7,True,True


In [45]:
recommendation_coverage_df.to_csv(
    REPORT_DIR / "Recommendation_Coverage_Check.csv",
    index=False
)